# PlasticWise AI - 03_finetune_and_error_analysis
**Week 4:** MobileNetV2 fine-tuning, confusion matrix, error analysis, optional experiments aur optional model comparison (VGG16 / ResNet50).

**Pehle:** Runtime > Change runtime type > **T4 GPU**. Week 3 ka kaam Drive mein hona chahiye, khas kar `models/mobilenetv2_frozen_best.keras`.
Cells upar se neeche ek ek karke run karo. **Cell 8 aur Cell 11 optional hain.** Test set is notebook mein bilkul use nahi hota.

## Cell 1: Setup (Drive + dataset dobara)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json
from pathlib import Path

BASE = '/content/drive/MyDrive/PlasticWise'
os.makedirs(f'{BASE}/results/week4', exist_ok=True)
os.makedirs(f'{BASE}/models', exist_ok=True)

import kagglehub
path = kagglehub.dataset_download("sumn2u/garbage-classification-v2")

IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def find_data_dir(root):
    best, best_n = None, 0
    for d in [Path(root)] + [p for p in Path(root).rglob('*') if p.is_dir()]:
        classes = [s for s in d.iterdir() if s.is_dir() and any(f.suffix.lower() in IMG_EXT for f in s.iterdir() if f.is_file())]
        if len(classes) > best_n:
            best, best_n = d, len(classes)
    return best

DATA_DIR = find_data_dir(path)
print("DATA_DIR =", DATA_DIR)

## Cell 2: Settings, CSVs, class weights load karo

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

print("TensorFlow:", tf.__version__, "| GPU:", tf.config.list_physical_devices('GPU'))

SEED = 42
IMG_SIZE = 224
BATCH = 32
WEIGHTS = 'imagenet'
EPOCHS_FT = 15       # fine-tuning ke max epochs
EPOCHS_CMP = 10      # optional model comparison ke epochs
tf.keras.utils.set_random_seed(SEED)

train = pd.read_csv(f'{BASE}/data/train.csv')
val   = pd.read_csv(f'{BASE}/data/val.csv')

for d in (train, val):
    if not Path(d.loc[0, 'filepath']).exists():
        d['filepath'] = d['relpath'].apply(lambda r: str(DATA_DIR / r))

class_names = json.load(open(f'{BASE}/data/class_names.json'))
class_weights = {int(k): v for k, v in json.load(open(f'{BASE}/data/class_weights.json')).items()}
NUM_CLASSES = len(class_names)
class_to_idx = {c: i for i, c in enumerate(class_names)}
y_train = train['label'].map(class_to_idx).values
y_val   = val['label'].map(class_to_idx).values

print(f"train {len(train)} | val {len(val)} | classes {NUM_CLASSES}")
print("Frozen model (Week 3) mojood hai:", os.path.exists(f'{BASE}/models/mobilenetv2_frozen_best.keras'))

## Cell 3: Input pipeline (Week 3 jaisa)
Test set ka pipeline jaan-boojh kar nahi banaya, taake galti se use na ho.

In [ ]:
from tensorflow.keras import layers
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

AUTOTUNE = tf.data.AUTOTUNE

def load_and_resize(path):
    img = tf.io.read_file(path)
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    return tf.cast(tf.round(img), tf.uint8)

def make_base_ds(df):
    return (tf.data.Dataset.from_tensor_slices(df['filepath'].values)
            .map(load_and_resize, num_parallel_calls=AUTOTUNE).cache())

def labels_ds(y):
    return tf.data.Dataset.from_tensor_slices(tf.one_hot(y, NUM_CLASSES))

augment_base = tf.keras.Sequential([
    layers.RandomFlip('horizontal'), layers.RandomRotation(0.1), layers.RandomZoom(0.1),
    layers.RandomContrast(0.1), layers.RandomBrightness(0.1)], name='augment_base')

augment_strong = tf.keras.Sequential([
    layers.RandomFlip('horizontal'), layers.RandomRotation(0.2), layers.RandomZoom(0.2),
    layers.RandomTranslation(0.1, 0.1), layers.RandomContrast(0.2),
    layers.RandomBrightness(0.2)], name='augment_strong')

base_train = make_base_ds(train)
base_val   = make_base_ds(val)

def make_train_ds(aug='base'):
    a = augment_strong if aug == 'strong' else augment_base
    def f(x, y):
        return preprocess_input(a(tf.cast(x, tf.float32), training=True)), y
    return (tf.data.Dataset.zip((base_train, labels_ds(y_train)))
            .shuffle(2000, seed=SEED).batch(BATCH)
            .map(f, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE))

val_ds = (tf.data.Dataset.zip((base_val, labels_ds(y_val))).batch(BATCH)
          .map(lambda x, y: (preprocess_input(tf.cast(x, tf.float32)), y), num_parallel_calls=AUTOTUNE)
          .prefetch(AUTOTUNE))
print("Pipeline ready.")

## Cell 4: Fine-tuning function aur helpers
`run_finetune(...)` Week 3 ka frozen model uthata hai, base ki **aakhri 30 layers** unfreeze karta hai (BatchNorm frozen), `Adam(1e-5)` se train karta hai, aur har experiment ko `experiments.csv` mein likhta hai. Har dafa frozen model se fresh start hota hai, isliye experiments barabar ke hain.

In [ ]:
import time, shutil
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, CSVLogger, ReduceLROnPlateau

W4 = f'{BASE}/results/week4'
os.makedirs(W4, exist_ok=True)
os.makedirs(f'{BASE}/models', exist_ok=True)
FROZEN_PATH = f'{BASE}/models/mobilenetv2_frozen_best.keras'
EXP_CSV = f'{W4}/experiments.csv'
experiments = []

def eval_val(model):
    probs = model.predict(val_ds, verbose=0)
    pred = probs.argmax(axis=1)
    return probs, pred, accuracy_score(y_val, pred), f1_score(y_val, pred, average='macro')

def log_experiment(row):
    experiments.append(row)
    pd.DataFrame(experiments).to_csv(EXP_CSV, index=False)

# Week 3 frozen model ka reference score
_m = tf.keras.models.load_model(FROZEN_PATH)
_, _, a0, f0 = eval_val(_m)
log_experiment({'tag': 'frozen_week3', 'unfreeze_last': 0, 'lr': 1e-3, 'label_smoothing': 0.0,
                'aug': 'base', 'epochs_run': None, 'best_epoch': None,
                'val_accuracy': round(a0, 4), 'val_macro_f1': round(f0, 4), 'train_seconds': None})
print(f"Frozen baseline (Week 3): acc {a0:.4f} | macro-F1 {f0:.4f}")

def run_finetune(tag, unfreeze_last=30, lr=1e-5, label_smoothing=0.0, aug='base', epochs=None):
    epochs = epochs or EPOCHS_FT
    model = tf.keras.models.load_model(FROZEN_PATH)
    base = [l for l in model.layers if isinstance(l, tf.keras.Model)][0]
    base.trainable = True
    for l in base.layers[:-unfreeze_last]:
        l.trainable = False
    for l in base.layers:                       # BatchNorm hamesha frozen
        if isinstance(l, layers.BatchNormalization):
            l.trainable = False
    model.compile(optimizer=tf.keras.optimizers.Adam(lr),
                  loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=label_smoothing),
                  metrics=['accuracy'])
    n_tr = int(sum(np.prod(w.shape) for w in model.trainable_weights))
    print(f"[{tag}] trainable params: {n_tr:,} | unfreeze_last={unfreeze_last} lr={lr} ls={label_smoothing} aug={aug}")

    cbs = [EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
           ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=2, verbose=1),
           ModelCheckpoint(f'{BASE}/models/{tag}.keras', monitor='val_loss', save_best_only=True),
           CSVLogger(f'{W4}/{tag}_log.csv')]
    t0 = time.time()
    hist = model.fit(make_train_ds(aug), validation_data=val_ds, epochs=epochs,
                     class_weight=class_weights, callbacks=cbs)
    secs = time.time() - t0
    _, _, acc, f1 = eval_val(model)
    h = pd.DataFrame(hist.history)
    log_experiment({'tag': tag, 'unfreeze_last': unfreeze_last, 'lr': lr,
                    'label_smoothing': label_smoothing, 'aug': aug,
                    'epochs_run': len(h), 'best_epoch': int(h['val_loss'].idxmin()) + 1,
                    'val_accuracy': round(acc, 4), 'val_macro_f1': round(f1, 4),
                    'train_seconds': round(secs)})
    print(f"[{tag}] DONE -> val accuracy {acc:.4f} | macro-F1 {f1:.4f} | {secs/60:.1f} min")
    return model, h

def promote(tag):
    # Chosen model ko final naam do + app ke liye info files likho
    shutil.copy(f'{BASE}/models/{tag}.keras', f'{BASE}/models/plasticwise_mobilenetv2.keras')
    json.dump(class_names, open(f'{BASE}/models/class_names.json', 'w'), indent=2)
    info = {'source_run': tag, 'input_size': IMG_SIZE,
            'preprocess': 'tf.keras.applications.mobilenet_v2.preprocess_input (0-255 -> -1..1)',
            'class_names': class_names}
    json.dump(info, open(f'{BASE}/models/model_info.json', 'w'), indent=2)
    print(f"Promoted '{tag}' -> models/plasticwise_mobilenetv2.keras (+ class_names.json, model_info.json)")

## Cell 5: Main fine-tuning run (aakhri 30 layers, lr 1e-5, max 15 epochs)
EarlyStopping aur ReduceLROnPlateau lage hain, isliye ye pehle bhi ruk sakta hai. Kuch minute lagte hain.

In [ ]:
model_ft, hist_ft = run_finetune('finetune30')
promote('finetune30')

## Cell 6: Combined curves (phase 1 frozen + phase 2 fine-tuning)

In [ ]:
p1_path = f'{BASE}/results/week3/frozen_training_log.csv'
p1 = pd.read_csv(p1_path) if os.path.exists(p1_path) else pd.DataFrame()
n1 = len(p1)
def cat(col): return list(p1[col]) + list(hist_ft[col]) if n1 else list(hist_ft[col])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (a, b, title) in zip(axes, [('accuracy', 'val_accuracy', 'Accuracy'), ('loss', 'val_loss', 'Loss')]):
    ax.plot(range(1, len(cat(a)) + 1), cat(a), label='train')
    ax.plot(range(1, len(cat(b)) + 1), cat(b), label='val')
    if n1: ax.axvline(n1 + 0.5, color='gray', linestyle='--', label='fine-tuning starts')
    ax.set_title(title); ax.set_xlabel('epoch'); ax.legend()
plt.tight_layout()
plt.savefig(f'{W4}/combined_curves.png', dpi=200)
plt.show()

## Cell 7: Error analysis (validation set)
Classification report, confusion matrix, sab se zyada confuse hone wali jodiyan, Plastic par focus, aur sab se **confident ghalat** images. `analyze(...)` function baad mein dusre model par bhi chalta hai.

In [ ]:
import seaborn as sns

def analyze(model, tag, show_n=12):
    probs, pred, acc, f1 = eval_val(model)
    print(f"[{tag}] validation accuracy {acc:.4f} | macro-F1 {f1:.4f}\n")

    rep_txt = classification_report(y_val, pred, labels=range(NUM_CLASSES), target_names=class_names, digits=3, zero_division=0)
    print(rep_txt)
    rep = classification_report(y_val, pred, labels=range(NUM_CLASSES), target_names=class_names, output_dict=True, zero_division=0)
    pd.DataFrame(rep).T.to_csv(f'{W4}/classification_report_{tag}.csv')

    cm = confusion_matrix(y_val, pred, labels=range(NUM_CLASSES))
    cm_norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)
    fig, axes = plt.subplots(1, 2, figsize=(18, 7))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names, ax=axes[0])
    axes[0].set_title('Confusion matrix (counts)')
    sns.heatmap(cm_norm * 100, annot=True, fmt='.0f', cmap='Blues', xticklabels=class_names, yticklabels=class_names, ax=axes[1])
    axes[1].set_title('Confusion matrix (% of true class)')
    for ax in axes: ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    plt.tight_layout(); plt.savefig(f'{W4}/confusion_matrix_{tag}.png', dpi=200); plt.show()

    pairs = pd.DataFrame([(class_names[i], class_names[j], int(cm[i, j]), round(cm_norm[i, j] * 100, 1))
                          for i in range(NUM_CLASSES) for j in range(NUM_CLASSES) if i != j and cm[i, j] > 0],
                         columns=['true', 'predicted_as', 'count', 'percent_of_true'])
    pairs = pairs.sort_values('count', ascending=False).reset_index(drop=True)
    pairs.to_csv(f'{W4}/top_confusions_{tag}.csv', index=False)
    print("\nSab se zyada confuse hone wali jodiyan (top 10):")
    print(pairs.head(10).to_string(index=False))

    if 'plastic' in class_to_idx:
        pi = class_to_idx['plastic']
        rec = cm[pi, pi] / max(cm[pi].sum(), 1); prec = cm[pi, pi] / max(cm[:, pi].sum(), 1)
        print(f"\nPLASTIC: recall {rec:.3f} | precision {prec:.3f}")
        print("Plastic ko in classes mein galat predict kiya:")
        print(pairs[pairs['true'] == 'plastic'].head(3).to_string(index=False))
        print("In classes ko Plastic predict kiya:")
        print(pairs[pairs['predicted_as'] == 'plastic'].head(3).to_string(index=False))

    print("\n===== COPY FROM HERE (ChatGPT/Claude error-analysis prompt ke liye) =====")
    print(rep_txt)
    print(pd.DataFrame(cm, index=class_names, columns=class_names).to_string())
    print("===== COPY TILL HERE =====")

    conf = probs[np.arange(len(pred)), pred]
    wrong = np.where(pred != y_val)[0]
    err = pd.DataFrame({'filepath': val['filepath'].values[wrong],
                        'true': [class_names[i] for i in y_val[wrong]],
                        'predicted': [class_names[i] for i in pred[wrong]],
                        'confidence': conf[wrong].round(3)}).sort_values('confidence', ascending=False)
    err.to_csv(f'{W4}/val_errors_{tag}.csv', index=False)
    print(f"\nKul ghalat predictions: {len(err)} / {len(val)}")

    def grid(df, title, fname, n):
        df = df.head(n)
        if len(df) == 0: return
        cols = 4; rows = int(np.ceil(len(df) / cols))
        fig, axes = plt.subplots(rows, cols, figsize=(14, 3.6 * rows)); axes = np.atleast_1d(axes).ravel()
        for ax in axes: ax.axis('off')
        for ax, (_, r) in zip(axes, df.iterrows()):
            ax.imshow(Image.open(r['filepath']).convert('RGB'))
            ax.set_title(f"True: {r['true']}\nPred: {r['predicted']} ({r['confidence']:.0%})", fontsize=10)
        plt.suptitle(title, fontsize=13); plt.tight_layout()
        plt.savefig(f'{W4}/{fname}_{tag}.png', dpi=150); plt.show()

    from PIL import Image
    grid(err, 'Sab se confident ghalat predictions', 'errors_confident', show_n)
    pl = err[(err['true'] == 'plastic') | (err['predicted'] == 'plastic')]
    grid(pl, 'Plastic se judi ghaltiyan', 'errors_plastic', 8)
    return {'acc': acc, 'f1': f1, 'pairs': pairs, 'errors': err}

res_ft = analyze(model_ft, 'finetune30')

### Error analysis kaise karna hai (roadmap wala prompt)
1. Upar **COPY FROM HERE ... COPY TILL HERE** ke beech ka text copy karo.
2. Ye prompt ChatGPT ya Claude mein paste karo, aur `[PASTE ...]` ki jagah wo text rakh do:

```
Here are my classification report and confusion matrix (pasted below). Explain in simple language which classes are confused, especially Plastic vs Paper, Glass, Metal and Trash. Suggest 5 practical improvements I can try in one week on Colab free GPU (for example stronger augmentation, class weights, higher resolution, label smoothing, removing bad images). Rank them by effort vs expected gain. Do not promise any accuracy number.
[PASTE REPORT AND MATRIX HERE]
```
3. Report ke liye **2 example ghaltiyan** `errors_confident_*.png` aur `errors_plastic_*.png` se chuno. Dekho ke kya label ghalat lag raha hai, ya image waqai mushkil hai (jaise plastic ki botal par paper label).
4. Roadmap kehta hai **sirf 1-2 fixes** try karo, sab nahi. Agla optional cell ek example experiment hai.

## Cell 8: (Optional) Experiment - stronger augmentation + label smoothing
Ye sirf tab chalao jab error analysis ke baad lage ke model overfit kar raha hai ya confusion ziyada hai. Dusre variants ke liye neeche comment wali lines try kar sakte ho (aik waqt mein 1-2 hi).

In [ ]:
# Experiment A: zyada augmentation + label smoothing 0.1
model_x, hist_x = run_finetune('finetune30_strongaug_ls', unfreeze_last=30, lr=1e-5,
                               label_smoothing=0.1, aug='strong')

# Experiment B (agar chaho): aakhri 50 layers unfreeze
# model_y, hist_y = run_finetune('finetune50', unfreeze_last=50, lr=1e-5)

## Cell 9: Sab experiments ka table, best model chuno (validation macro-F1 se)

In [ ]:
exp = pd.DataFrame(experiments)
print(exp.to_string(index=False))

cand = exp[exp['tag'] != 'frozen_week3'].sort_values('val_macro_f1', ascending=False)
best_tag = cand.iloc[0]['tag']
print("\nBest (val macro-F1):", best_tag)

best_model = tf.keras.models.load_model(f'{BASE}/models/{best_tag}.keras')
promote(best_tag)
res_best = analyze(best_model, best_tag)

**Note:** Agar sirf `finetune30` chala hai to best wahi hoga, aur ye cell wahi dobara dikhayega. Model chunna validation par hota hai, test set par nahi.

## Cell 10: Saved model ka sanity check (Week 5 app jaisa tareeqa)
App mein bhi bilkul yehi 3 steps hongay: 224x224 resize, `preprocess_input`, `model.predict`.

In [ ]:
final = tf.keras.models.load_model(f'{BASE}/models/plasticwise_mobilenetv2.keras')
names = json.load(open(f'{BASE}/models/class_names.json'))

row = val.iloc[0]
img = tf.keras.utils.load_img(row['filepath'], target_size=(IMG_SIZE, IMG_SIZE))
arr = preprocess_input(np.expand_dims(tf.keras.utils.img_to_array(img), 0))
p = final.predict(arr, verbose=0)[0]
top = p.argsort()[::-1][:3]
print("True label:", row['label'])
for i in top:
    print(f"  {names[i]:10s} {p[i]:.1%}")
print("\nModel file size: %.1f MB" % (os.path.getsize(f'{BASE}/models/plasticwise_mobilenetv2.keras') / 1e6))

## Cell 11: (Optional) Model comparison - MobileNetV2 vs VGG16 (vs ResNet50)
Wahi split, wahi augmentation, wahi head (`GlobalAveragePooling2D + Dropout(0.3) + Dense`), wahi training settings (base frozen, Adam 1e-3, `EPOCHS_CMP` epochs, class weights, EarlyStopping). Har model ka **apna `preprocess_input`** model ke andar lagta hai, isliye comparison fair hai. Ye **frozen** comparison hai (fine-tuned MobileNetV2 se nahi).

VGG16 aur ResNet50 bhari hain, isliye time zyada lagta hai. Sirf wo models rakho jo chahiye.

In [ ]:
from tensorflow.keras.applications import MobileNetV2, VGG16, ResNet50
from tensorflow.keras.applications import mobilenet_v2, vgg16, resnet50

MODELS_TO_COMPARE = ['MobileNetV2', 'VGG16']        # ResNet50 bhi chahiye to list mein daal do

CFG = {'MobileNetV2': (MobileNetV2, mobilenet_v2.preprocess_input),
       'VGG16': (VGG16, vgg16.preprocess_input),
       'ResNet50': (ResNet50, resnet50.preprocess_input)}

def build_model(name):
    ctor, prep = CFG[name]
    base = ctor(include_top=False, weights=WEIGHTS, input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = False
    inp = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = layers.Lambda(prep, name='preprocess')(inp)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(NUM_CLASSES, activation='softmax')(x)
    m = tf.keras.Model(inp, out, name=f'cmp_{name}')
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='categorical_crossentropy', metrics=['accuracy'])
    return m

# Raw (0-255) data; har model apna preprocess khud karta hai
cmp_train = (tf.data.Dataset.zip((base_train, labels_ds(y_train))).shuffle(2000, seed=SEED).batch(BATCH)
             .map(lambda x, y: (augment_base(tf.cast(x, tf.float32), training=True), y), num_parallel_calls=AUTOTUNE)
             .prefetch(AUTOTUNE))
cmp_val = (tf.data.Dataset.zip((base_val, labels_ds(y_val))).batch(BATCH)
           .map(lambda x, y: (tf.cast(x, tf.float32), y), num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE))

rows = []
for name in MODELS_TO_COMPARE:
    m = build_model(name)
    t0 = time.time()
    m.fit(cmp_train, validation_data=cmp_val, epochs=EPOCHS_CMP, class_weight=class_weights,
          callbacks=[EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)], verbose=2)
    secs = time.time() - t0
    pred = m.predict(cmp_val, verbose=0).argmax(axis=1)
    rows.append({'model': name, 'val_accuracy': round(accuracy_score(y_val, pred), 4),
                 'val_macro_f1': round(f1_score(y_val, pred, average='macro'), 4),
                 'train_minutes': round(secs / 60, 1),
                 'total_params_M': round(m.count_params() / 1e6, 2),
                 'trainable_params_K': round(sum(int(np.prod(w.shape)) for w in m.trainable_weights) / 1e3, 1)})
    print(rows[-1])

cmp_df = pd.DataFrame(rows)
print(cmp_df.to_string(index=False))
cmp_df.to_csv(f'{W4}/model_comparison.csv', index=False)

ax = cmp_df.set_index('model')[['val_accuracy', 'val_macro_f1']].plot.bar(figsize=(8, 5), rot=0)
ax.set_ylim(0, 1); ax.set_title('Frozen backbones: validation accuracy and macro-F1')
plt.tight_layout(); plt.savefig(f'{W4}/model_comparison.png', dpi=200); plt.show()

## Week 4 checklist
- [ ] `models/plasticwise_mobilenetv2.keras` aur `models/class_names.json` Drive mein hain
- [ ] `combined_curves.png`, `confusion_matrix_*.png`, `errors_*.png` bane (`results/week4/`)
- [ ] Error-analysis ka aadha page likha, 2 example ghaltiyon ke saath
- [ ] `experiments.csv` mein frozen vs fine-tuned (aur agar kiya to experiment) ka table hai
- [ ] (Optional) `model_comparison.csv` bana
- [ ] Notebook, `class_names.json`, model file (chhota hai, ~10 MB) GitHub par commit

**Week 5 ke liye yaad rakho:** model ko `preprocess_input` wali (-1 se 1) images chahiye. Ye `models/model_info.json` mein bhi likha hai. Test set par evaluation Week 5 mein, sirf final model par, ek hi dafa.